In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

TAG = "v2"
MESES_GEC = [5, 6, 7, 8]
UMBRAL_TARGET_DEFECTO = 50.0
UMBRAL_TARGET = {"Aysén": 80.0}
K_PRESUPUESTO = 0.30
CONFIG = {
    "Santiago": {"desde": None, "fecha_test": "2025-01-01"},
    "Temuco": {"desde": None, "fecha_test": "2025-01-01"},
    "Aysén": {"desde": None, "fecha_test": "2025-01-01"},
    "Valdivia": {"desde": "2025-04-01", "fecha_test": "2026-01-01"},
}

# Variante "observado": delta = pronóstico noche de mañana - noche de hoy OBSERVADA (ERA5)
FEATS_OBS = [
    "temp_min_day_forecast",
    "wind_mean_day_forecast",
    "temperature_2m_noche_forecast",
    "wind_speed_10m_noche_forecast",
    "delta_temp_noche",
    "delta_wind_noche",
    "estancamiento_manana",
]
# Variante "GFS": delta = pronóstico noche de mañana - pronóstico noche de hoy (como en vivo)
FEATS_GFS = [
    f.replace("delta_temp_noche", "delta_temp_gfs").replace("delta_wind_noche", "delta_wind_gfs")
    for f in FEATS_OBS
]

gold_name = f"daily_modeling_gold_{TAG}.parquet"
gold_path = Path("data/processed") / gold_name
if not gold_path.exists():
  gold_path = Path("../data/processed") / gold_name
g = pd.read_parquet(gold_path)
g["date_local"] = pd.to_datetime(g["date_local"])

# El pronóstico de la noche de HOY es el que en la fila de AYER era "el de mañana".
ayer = g[["city", "date_local", "temperature_2m_noche_forecast",
          "wind_speed_10m_noche_forecast"]].copy()
ayer["date_local"] += pd.Timedelta(days=1)
ayer = ayer.rename(columns={
    "temperature_2m_noche_forecast": "temp_noche_hoy_gfs",
    "wind_speed_10m_noche_forecast": "wind_noche_hoy_gfs",
})
g = g.merge(ayer, on=["city", "date_local"], how="left")
g["delta_temp_gfs"] = g["temperature_2m_noche_forecast"] - g["temp_noche_hoy_gfs"]
g["delta_wind_gfs"] = g["wind_speed_10m_noche_forecast"] - g["wind_noche_hoy_gfs"]


def modelo():
  return make_pipeline(
      StandardScaler(),
      LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000),
  )


print("=" * 78)
print("  1) ¿CUÁNTO DIFIEREN LOS DOS DELTAS? (invierno, días de test)")
print("=" * 78)
filas = []
for ciudad, cfg in CONFIG.items():
  d = g[(g["city"] == ciudad) & g["month"].isin(MESES_GEC)]
  if cfg["desde"]:
    d = d[d["date_local"] >= cfg["desde"]]
  umbral_t = UMBRAL_TARGET.get(ciudad, UMBRAL_TARGET_DEFECTO)
  d = d.assign(y=(d["pm25_next_day"] >= umbral_t).astype(int))
  d = d.dropna(subset=FEATS_OBS + FEATS_GFS).reset_index(drop=True)

  tr = d[d["date_local"] < cfg["fecha_test"]]
  te = d[d["date_local"] >= cfg["fecha_test"]]
  if tr["y"].sum() < 10 or te["y"].sum() == 0 or len(te) == 0:
    print(f"[{ciudad}] datos insuficientes, se omite.")
    continue

  dif_t = te["delta_temp_noche"] - te["delta_temp_gfs"]
  dif_w = te["delta_wind_noche"] - te["delta_wind_gfs"]
  print(
      f"[{ciudad}] temp: sesgo {dif_t.mean():+.2f} °C, desv. {dif_t.std():.2f} "
      f"(el delta mismo tiene desv. {te['delta_temp_noche'].std():.2f}) | "
      f"viento: sesgo {dif_w.mean():+.2f} km/h, desv. {dif_w.std():.2f} "
      f"(el delta mismo tiene desv. {te['delta_wind_noche'].std():.2f})"
  )

  # A) Referencia: entrena y prueba con delta observado (lo que reportamos)
  m_obs = modelo().fit(tr[FEATS_OBS], tr["y"])
  p_obs = m_obs.predict_proba(te[FEATS_OBS])[:, 1]
  thr = np.quantile(m_obs.predict_proba(tr[FEATS_OBS])[:, 1], 1 - K_PRESUPUESTO)

  # B) Desajuste: modelo entrenado con delta observado, usado con delta GFS (lo que hace el script en vivo)
  te_gfs = te[FEATS_GFS].copy()
  te_gfs.columns = FEATS_OBS
  p_mis = m_obs.predict_proba(te_gfs)[:, 1]

  # C) Corregido: entrena y prueba con delta GFS-GFS
  m_gfs = modelo().fit(tr[FEATS_GFS], tr["y"])
  p_gfs = m_gfs.predict_proba(te[FEATS_GFS])[:, 1]
  thr_gfs = np.quantile(m_gfs.predict_proba(tr[FEATS_GFS])[:, 1], 1 - K_PRESUPUESTO)

  filas.append({
      "ciudad": ciudad,
      "n_test": len(te),
      "AUC_A_obs": round(roc_auc_score(te["y"], p_obs), 3),
      "AUC_B_desajuste": round(roc_auc_score(te["y"], p_mis), 3),
      "AUC_C_gfs_reentrenado": round(roc_auc_score(te["y"], p_gfs), 3),
      "%alertas_A": round(float((p_obs >= thr).mean()) * 100, 1),
      "%alertas_B": round(float((p_mis >= thr).mean()) * 100, 1),
      "%alertas_C": round(float((p_gfs >= thr_gfs).mean()) * 100, 1),
  })

print("\n" + "=" * 78)
print("  2) AUC Y % DE DÍAS ALERTADOS EN EL TEST")
print("=" * 78)
res = pd.DataFrame(filas)
print(res.to_string(index=False))
print(
    "\nA = como se evaluó (delta observado).  B = modelo A usado con delta GFS (script en vivo).\n"
    "C = modelo reentrenado con delta GFS-GFS.\n"
    "Si A y B casi coinciden en AUC y en % de alertas, el desajuste pesa poco.\n"
    "Si B se aleja de A, conviene aplicar el cambio al ETL (opción C)."
)
res.to_csv(gold_path.resolve().parent / f"sensibilidad_delta_{TAG}.csv", index=False)

  1) ¿CUÁNTO DIFIEREN LOS DOS DELTAS? (invierno, días de test)
[Santiago] temp: sesgo +2.89 °C, desv. 1.60 (el delta mismo tiene desv. 2.56) | viento: sesgo +2.21 km/h, desv. 1.68 (el delta mismo tiene desv. 2.31)
[Temuco] temp: sesgo +0.13 °C, desv. 1.01 (el delta mismo tiene desv. 2.25) | viento: sesgo -2.59 km/h, desv. 2.45 (el delta mismo tiene desv. 6.75)
[Aysén] temp: sesgo -0.86 °C, desv. 1.38 (el delta mismo tiene desv. 2.41) | viento: sesgo +1.69 km/h, desv. 1.38 (el delta mismo tiene desv. 2.33)
[Valdivia] temp: sesgo -0.34 °C, desv. 1.08 (el delta mismo tiene desv. 2.23) | viento: sesgo -2.61 km/h, desv. 3.00 (el delta mismo tiene desv. 4.64)

  2) AUC Y % DE DÍAS ALERTADOS EN EL TEST
  ciudad  n_test  AUC_A_obs  AUC_B_desajuste  AUC_C_gfs_reentrenado  %alertas_A  %alertas_B  %alertas_C
Santiago     240      0.768            0.620                  0.618        54.6        14.2        30.8
  Temuco     223      0.860            0.850                  0.852        27.4        

In [2]:
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yaml
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ------------------------------------------------------------------------------
# CONFIGURACIÓN Y RUTAS
# ------------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("AirQualityETL")

CURRENT_DIR = Path().resolve()
BASE_DIR = (
    CURRENT_DIR
    if (CURRENT_DIR / "stations.yaml").exists()
    else CURRENT_DIR.parent
)

YAML_CONFIG_PATH = BASE_DIR / "stations.yaml"
SINCA_DIR = BASE_DIR / "Datos Sinca"
RAW_WEATHER_DIR = BASE_DIR / "data" / "raw" / "weather"
RAW_FORECAST_DIR = BASE_DIR / "data" / "raw" / "forecast"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_WEATHER_DIR.mkdir(parents=True, exist_ok=True)
RAW_FORECAST_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GLOBAL_START_DATE = "2020-01-01"
GLOBAL_END_DATE = "2026-10-14"

# Sufijo de versión: evita reemplazar los parquet generados por la versión anterior.
OUTPUT_TAG = "prueba_final"
# Umbral de alerta sobre el promedio diario de PM2.5 (µg/m³).
UMBRAL_ALERTA = 50.0
# Mínimo de horas válidas para aceptar un promedio diario.
MIN_HORAS_DIA = 18

logger.info(f"Directorio base detectado: {BASE_DIR}")
logger.info(f"Ventana analítica: {GLOBAL_START_DATE} a {GLOBAL_END_DATE}")
logger.info(f"Versión de salida: {OUTPUT_TAG}")


# ------------------------------------------------------------------------------
# 1. FUNCIONES AUXILIARES DE PROCESAMIENTO Y DESCARGA
# ------------------------------------------------------------------------------
def parse_sinca_csv(
    filepath: Path,
    timezone: str = "America/Santiago",
    max_pm25: float = 1200.0,
) -> pd.DataFrame:
  """Lee y limpia el CSV del SINCA con soporte para Aysén Dinámico."""
  df = pd.read_csv(
      filepath, sep=";", decimal=",", encoding="latin1", low_memory=False
  )
  df.columns = [str(c).strip() for c in df.columns]

  col_fecha = next((c for c in df.columns if "FECHA" in c.upper()), None)
  col_hora = next((c for c in df.columns if "HORA" in c.upper()), None)

  if not col_fecha or not col_hora:
    raise KeyError(
        f"Estructura inválida en {filepath.name}: Faltan columnas de fecha u"
        " hora."
    )

  fecha_clean = (
      df[col_fecha]
      .astype(str)
      .str.strip()
      .str.replace(r"\.0$", "", regex=True)
      .str.zfill(6)
  )
  hora_clean = (
      df[col_hora]
      .astype(str)
      .str.strip()
      .str.replace(r"\.0$", "", regex=True)
      .str.zfill(4)
  )

  hora_num = pd.to_numeric(hora_clean, errors="coerce")
  hora_anterior = hora_num.shift(1)
  fecha_anterior = fecha_clean.shift(1)
  es_cierre_dia = (
      (hora_num == 0) & (hora_anterior == 2300) & (fecha_clean == fecha_anterior)
  )

  n_cierres = es_cierre_dia.sum()
  logger.info(
      f"[{filepath.name}] Horas '0' corregidas como cierre de día: {n_cierres}"
  )

  df["time_local"] = pd.to_datetime(
      fecha_clean + " " + hora_clean, format="%y%m%d %H%M", errors="coerce"
  )

  if n_cierres > 0:
    df.loc[es_cierre_dia, "time_local"] += pd.Timedelta(days=1)

  df = df.dropna(subset=["time_local"]).sort_values("time_local").copy()

  if timezone == "America/Aysen_Dynamic":
    fecha_corte = pd.Timestamp("2025-04-05 23:00:00")
    mask_hist = df["time_local"] < fecha_corte

    df_hist = df[mask_hist].copy()
    df_hist["time_utc"] = (
        df_hist["time_local"]
        .dt.tz_localize(
            "America/Santiago", ambiguous="NaT", nonexistent="shift_forward"
        )
        .dt.tz_convert("UTC")
    )

    df_post = df[~mask_hist].copy()
    df_post["time_utc"] = (
        df_post["time_local"]
        .dt.tz_localize(
            "America/Punta_Arenas", ambiguous="NaT", nonexistent="shift_forward"
        )
        .dt.tz_convert("UTC")
    )

    df = pd.concat([df_hist, df_post])
  else:
    df["time_utc"] = (
        df["time_local"]
        .dt.tz_localize(
            timezone, ambiguous="NaT", nonexistent="shift_forward"
        )
        .dt.tz_convert("UTC")
    )

  col_val = next(
      (
          c
          for c in df.columns
          if "validados" in c.lower() and "no" not in c.lower()
      ),
      None,
  )
  col_pre = next((c for c in df.columns if "preliminares" in c.lower()), None)

  if not col_val and not col_pre:
    raise KeyError(
        f"El archivo {filepath.name} no contiene columnas válidas de PM2.5."
    )

  df["pm2_5"] = np.nan
  df["data_status"] = "no_medido"

  if col_val:
    s_val = pd.to_numeric(df[col_val], errors="coerce")
    m_val = s_val.notna()
    df.loc[m_val, "pm2_5"] = s_val[m_val]
    df.loc[m_val, "data_status"] = "validado"

  if col_pre:
    s_pre = pd.to_numeric(df[col_pre], errors="coerce")
    m_pre = df["pm2_5"].isna() & s_pre.notna()
    df.loc[m_pre, "pm2_5"] = s_pre[m_pre]
    df.loc[m_pre, "data_status"] = "preliminar"

  invalid_mask = (df["pm2_5"] < 0) | (df["pm2_5"] > max_pm25)
  if invalid_mask.sum() > 0:
    logger.warning(
        f"[{filepath.name}] {invalid_mask.sum()} registros descartados por"
        f" límite físico (0 - {max_pm25} µg/m³)."
    )
    df.loc[invalid_mask, "pm2_5"] = np.nan
    df.loc[invalid_mask, "data_status"] = "anomalia_descartada"

  df_clean = df.dropna(subset=["time_utc"]).drop_duplicates(subset=["time_utc"])
  return (
      df_clean[["time_utc", "pm2_5", "data_status"]]
      .sort_values("time_utc")
      .reset_index(drop=True)
  )


def get_weather_session() -> requests.Session:
  """Sesión HTTP con reintentos exponenciales automáticos."""
  session = requests.Session()
  retries = Retry(
      total=5,
      backoff_factor=1.5,
      status_forcelist=[429, 500, 502, 503, 504],
      raise_on_status=True,
  )
  session.mount("https://", HTTPAdapter(max_retries=retries))
  return session


def trim_trailing_nulls(df: pd.DataFrame, col: str) -> pd.DataFrame:
  """Recorta las últimas filas sin dato (ERA5 tiene unos días de retraso)."""
  last_valid = df[col].last_valid_index()
  if last_valid is None:
    return df.iloc[0:0]
  return df.loc[:last_valid]


def fetch_weather_cached(
    station_id: str,
    lat: float,
    lon: float,
    start_date: str,
    end_date: str,
    cache_dir: Path,
) -> pd.DataFrame:
  """Descarga o lee en caché la meteorología ERA5 observada en UTC."""
  cache_file = (
      cache_dir / f"weather_{station_id}_{start_date}_{end_date}.parquet"
  )

  if cache_file.exists():
    logger.info(
        f"[{station_id}] Leyendo meteorología desde caché ({cache_file.name})."
    )
    return pd.read_parquet(cache_file)

  url = "https://archive-api.open-meteo.com/v1/archive"
  variables = [
      "temperature_2m",
      "relative_humidity_2m",
      "dew_point_2m",
      "wind_speed_10m",
      "wind_direction_10m",
      "surface_pressure",
      "precipitation",
      "cloud_cover",
      "direct_normal_irradiance",
  ]

  params = {
      "latitude": lat,
      "longitude": lon,
      "start_date": start_date,
      "end_date": end_date,
      "hourly": variables,
      "timezone": "UTC",
  }

  session = get_weather_session()
  logger.info(f"[{station_id}] Solicitando ERA5 a Open-Meteo...")
  response = session.get(url, params=params, timeout=50)
  response.raise_for_status()

  payload = response.json()
  df_weather = pd.DataFrame(payload["hourly"])
  df_weather["time_utc"] = pd.to_datetime(df_weather["time"]).dt.tz_localize(
      "UTC"
  )
  df_weather = df_weather.drop(columns=["time"]).sort_values("time_utc")

  df_weather.to_parquet(cache_file, index=False)
  logger.info(f"[{station_id}] Meteorología guardada en caché.")
  return df_weather


def fetch_forecast_cached(
    station_id: str,
    lat: float,
    lon: float,
    start_date: str,
    end_date: str,
    cache_dir: Path,
) -> pd.DataFrame:
  """Descarga o lee en caché las salidas de pronóstico numérico (GFS Seamless) en UTC."""
  cache_file = (
      cache_dir / f"forecast_gfs_{station_id}_{start_date}_{end_date}.parquet"
  )

  if cache_file.exists():
    logger.info(
        f"[{station_id}] Leyendo pronóstico GFS desde caché"
        f" ({cache_file.name})."
    )
    return pd.read_parquet(cache_file)

  url = "https://archive-api.open-meteo.com/v1/archive"
  params = {
      "latitude": lat,
      "longitude": lon,
      "start_date": start_date,
      "end_date": end_date,
      "hourly": ["temperature_2m", "wind_speed_10m", "precipitation"],
      "timezone": "UTC",
      "models": "gfs_seamless",
  }

  session = get_weather_session()
  logger.info(f"[{station_id}] Descargando pronósticos numéricos GFS...")
  response = session.get(url, params=params, timeout=60)
  response.raise_for_status()

  payload = response.json()
  df_fc = pd.DataFrame(payload["hourly"])
  df_fc["time_utc"] = pd.to_datetime(df_fc["time"]).dt.tz_localize("UTC")
  df_fc = df_fc.rename(
      columns={
          "temperature_2m": "temp_forecast",
          "wind_speed_10m": "wind_forecast",
          "precipitation": "precip_forecast",
      }
  ).drop(columns=["time"])

  df_fc = df_fc.sort_values("time_utc").reset_index(drop=True)
  df_fc.to_parquet(cache_file, index=False)
  logger.info(f"[{station_id}] Pronóstico GFS guardado en caché.")
  time.sleep(0.5)
  return df_fc


def flag_binaria(serie: pd.Series, umbral: float) -> pd.Series:
  """Bandera 0/1 que conserva <NA> donde no hay dato (NaN >= x NO es 0)."""
  return (serie >= umbral).astype("Int64").where(serie.notna())


# ------------------------------------------------------------------------------
# 2. BUCLE PRINCIPAL DE PROCESAMIENTO
# ------------------------------------------------------------------------------
with open(YAML_CONFIG_PATH, "r", encoding="utf-8") as f:
  config = yaml.safe_load(f)

consolidated_hourly_list = []
consolidated_daily_gold = []
forecast_quality = []

for st in config["stations"]:
  station_id = st["id"]
  station_name = st["name"]
  city = st["city"]
  lat = st["latitude"]
  lon = st["longitude"]
  tz = st.get("timezone", "America/Santiago")
  fname = st.get("filename") or f"m25_{station_id}.csv"
  csv_file = SINCA_DIR / fname

  logger.info("=" * 65)
  logger.info(f"Procesando: {station_name} ({city}) | Zona Horaria: {tz}")

  if not csv_file.exists():
    logger.error(f"Archivo {csv_file.name} no encontrado en {SINCA_DIR}. Se omite.")
    continue

  # 1. Parsear SINCA
  df_sinca = parse_sinca_csv(csv_file, timezone=tz)

  # 2. Meteorología observada (ERA5), sin los días finales aún vacíos
  df_weather = fetch_weather_cached(
      station_id=station_id,
      lat=lat,
      lon=lon,
      start_date=GLOBAL_START_DATE,
      end_date=GLOBAL_END_DATE,
      cache_dir=RAW_WEATHER_DIR,
  )
  df_weather = trim_trailing_nulls(df_weather, "temperature_2m")

  # 3. Pronóstico numérico (GFS)
  df_forecast = fetch_forecast_cached(
      station_id=station_id,
      lat=lat,
      lon=lon,
      start_date=GLOBAL_START_DATE,
      end_date=GLOBAL_END_DATE,
      cache_dir=RAW_FORECAST_DIR,
  )

  # 4. Unir grilla climática
  weather_and_fc = pd.merge(
      df_weather, df_forecast, on="time_utc", how="inner"
  )

  # 5. Cruce con SINCA
  merged = pd.merge(df_sinca, weather_and_fc, on="time_utc", how="right")
  merged["data_status"] = merged["data_status"].fillna("no_medido")

  merged["station_id"] = station_id
  merged["station_name"] = station_name
  merged["city"] = city

  # 5b. Diagnóstico: ¿cuánto se parece el "pronóstico" a la observación ERA5?
  mae_temp = (merged["temp_forecast"] - merged["temperature_2m"]).abs().mean()
  mae_wind = (merged["wind_forecast"] - merged["wind_speed_10m"]).abs().mean()
  forecast_quality.append({
      "city": city,
      "station_id": station_id,
      "mae_temp_C": round(float(mae_temp), 3),
      "mae_wind_kmh": round(float(mae_wind), 3),
  })
  logger.info(
      f"[{city}] Error medio pronóstico vs ERA5: temp={mae_temp:.2f} °C,"
      f" viento={mae_wind:.2f} km/h"
  )

  # 6. Descomposición temporal y time_local naive
  if tz == "America/Aysen_Dynamic":
    corte_utc = pd.Timestamp("2025-04-06 02:00:00+00:00")
    mask_hist = merged["time_utc"] < corte_utc

    dt_hist = merged.loc[mask_hist, "time_utc"].dt.tz_convert(
        "America/Santiago"
    )
    dt_post = merged.loc[~mask_hist, "time_utc"].dt.tz_convert(
        "America/Punta_Arenas"
    )

    merged.loc[mask_hist, "hour_local"] = dt_hist.dt.hour
    merged.loc[~mask_hist, "hour_local"] = dt_post.dt.hour
    merged.loc[mask_hist, "month"] = dt_hist.dt.month
    merged.loc[~mask_hist, "month"] = dt_post.dt.month
    merged.loc[mask_hist, "dayofweek"] = dt_hist.dt.dayofweek
    merged.loc[~mask_hist, "dayofweek"] = dt_post.dt.dayofweek

    merged["time_local"] = pd.concat([
        dt_hist.dt.tz_localize(None),
        dt_post.dt.tz_localize(None),
    ]).sort_index()
  else:
    dt_loc = merged["time_utc"].dt.tz_convert(tz)
    merged["hour_local"] = dt_loc.dt.hour
    merged["month"] = dt_loc.dt.month
    merged["dayofweek"] = dt_loc.dt.dayofweek
    merged["time_local"] = dt_loc.dt.tz_localize(None)

  merged["hour_local"] = merged["hour_local"].astype(int)
  merged["month"] = merged["month"].astype(int)
  merged["dayofweek"] = merged["dayofweek"].astype(int)
  merged["date_local"] = merged["time_local"].dt.floor("D")

  rad = np.radians(merged["wind_direction_10m"])
  merged["wind_u"] = -merged["wind_speed_10m"] * np.sin(rad)
  merged["wind_v"] = -merged["wind_speed_10m"] * np.cos(rad)

  consolidated_hourly_list.append(merged)

  # 7. Capa Gold diaria
  noche_mask = merged["hour_local"].between(18, 23)

  obs_daily = (
      merged.groupby("date_local")
      .agg(
          pm25_horas_validas=("pm2_5", "count"),
          pm25_mean_raw=("pm2_5", "mean"),
          pm25_max_raw=("pm2_5", "max"),
          temp_min_day=("temperature_2m", "min"),
          wind_mean_day=("wind_speed_10m", "mean"),
      )
      .reset_index()
  )
  # Promedio/máximo diario solo si hay horas suficientes
  dia_valido = obs_daily["pm25_horas_validas"] >= MIN_HORAS_DIA
  obs_daily["pm25_mean_day"] = obs_daily["pm25_mean_raw"].where(dia_valido)
  obs_daily["pm25_max_day"] = obs_daily["pm25_max_raw"].where(dia_valido)
  obs_daily = obs_daily.drop(columns=["pm25_mean_raw", "pm25_max_raw"])

  obs_noche = (
      merged[noche_mask]
      .groupby("date_local")
      .agg(
          temperature_2m_noche=("temperature_2m", "mean"),
          wind_speed_10m_noche=("wind_speed_10m", "mean"),
      )
      .reset_index()
  )

  fc_daily = (
      merged.groupby("date_local")
      .agg(
          fc_temp_min=("temp_forecast", "min"),
          fc_wind_mean=("wind_forecast", "mean"),
      )
      .reset_index()
  )

  fc_noche = (
      merged[noche_mask]
      .groupby("date_local")
      .agg(
          fc_temp_noche=("temp_forecast", "mean"),
          fc_wind_noche=("wind_forecast", "mean"),
      )
      .reset_index()
  )

  gold_station = obs_daily
  for parte in (obs_noche, fc_daily, fc_noche):
    gold_station = gold_station.merge(parte, on="date_local", how="left")

  # Calendario diario completo: garantiza que shift(±1) sea exactamente ±1 día
  gold_station = gold_station.sort_values("date_local")
  calendario = pd.date_range(
      gold_station["date_local"].min(),
      gold_station["date_local"].max(),
      freq="D",
  )
  gold_station = (
      gold_station.set_index("date_local")
      .reindex(calendario)
      .rename_axis("date_local")
      .reset_index()
  )
  gold_station["month"] = gold_station["date_local"].dt.month
  gold_station["dayofweek"] = gold_station["date_local"].dt.dayofweek

  # El pronóstico de MAÑANA (t+1) para decidir HOY (t)
  gold_station["temp_min_day_forecast"] = gold_station["fc_temp_min"].shift(-1)
  gold_station["wind_mean_day_forecast"] = gold_station["fc_wind_mean"].shift(-1)
  gold_station["temperature_2m_noche_forecast"] = gold_station[
      "fc_temp_noche"
  ].shift(-1)
  gold_station["wind_speed_10m_noche_forecast"] = gold_station[
      "fc_wind_noche"
  ].shift(-1)

  # Noche de HOY según el mismo pronóstico GFS (igual que en inferencia en vivo)
  gold_station["temp_noche_hoy_gfs"] = gold_station["fc_temp_noche"]
  gold_station["wind_noche_hoy_gfs"] = gold_station["fc_wind_noche"]

  gold_station = gold_station.drop(
      columns=["fc_temp_min", "fc_wind_mean", "fc_temp_noche", "fc_wind_noche"]
  )

  # DELTAS DEL MODELO: GFS contra GFS (noche de mañana - noche de hoy)
  gold_station["delta_temp_noche"] = (
      gold_station["temperature_2m_noche_forecast"]
      - gold_station["temp_noche_hoy_gfs"]
  )
  gold_station["delta_wind_noche"] = (
      gold_station["wind_speed_10m_noche_forecast"]
      - gold_station["wind_noche_hoy_gfs"]
  )
  # Solo referencia (NO son features): contra la noche de hoy observada (ERA5)
  gold_station["delta_temp_noche_obs"] = (
      gold_station["temperature_2m_noche_forecast"]
      - gold_station["temperature_2m_noche"]
  )
  gold_station["delta_wind_noche_obs"] = (
      gold_station["wind_speed_10m_noche_forecast"]
      - gold_station["wind_speed_10m_noche"]
  )
  gold_station["estancamiento_manana"] = gold_station[
      "wind_speed_10m_noche_forecast"
  ] / np.maximum(gold_station["temperature_2m_noche_forecast"] + 5.0, 1.0)

  # Persistencia: información disponible HOY al momento de decidir
  gold_station["pm25_lag1"] = gold_station["pm25_mean_day"].shift(1)
  gold_station["pm25_lag2"] = gold_station["pm25_mean_day"].shift(2)
  gold_station["pm25_roll3"] = (
      gold_station["pm25_mean_day"].rolling(3, min_periods=2).mean()
  )
  gold_station["pm25_roll7"] = (
      gold_station["pm25_mean_day"].rolling(7, min_periods=4).mean()
  )

  # Target (conserva <NA> cuando no hay promedio diario válido)
  gold_station["pm25_next_day"] = gold_station["pm25_mean_day"].shift(-1)
  gold_station["alerta_today"] = flag_binaria(
      gold_station["pm25_mean_day"], UMBRAL_ALERTA
  )
  gold_station["alerta_next_day"] = flag_binaria(
      gold_station["pm25_next_day"], UMBRAL_ALERTA
  )

  gold_station["station_id"] = station_id
  gold_station["city"] = city
  consolidated_daily_gold.append(gold_station)

  n_obj = int(gold_station["alerta_next_day"].notna().sum())
  n_pos = int(gold_station["alerta_next_day"].fillna(0).sum())
  logger.info(
      f"[{city}] Capa Gold: {len(gold_station):,} días | target válido:"
      f" {n_obj:,} | alertas (>= {UMBRAL_ALERTA:.0f}): {n_pos:,}"
  )

# ------------------------------------------------------------------------------
# 3. EXPORTACIÓN PARQUET (con sufijo de versión)
# ------------------------------------------------------------------------------
df_final_hourly = pd.concat(consolidated_hourly_list, ignore_index=True)
hourly_parquet = (
    PROCESSED_DIR
    / f"air_quality_chile_2020_2026_consolidated_{OUTPUT_TAG}.parquet"
)
df_final_hourly.to_parquet(hourly_parquet, index=False, engine="pyarrow")
logger.info(f"Parquet Horario Consolidado: {hourly_parquet}")

df_final_gold = pd.concat(consolidated_daily_gold, ignore_index=True)
df_final_gold = df_final_gold.dropna(
    subset=["alerta_next_day", "temp_min_day_forecast"]
).reset_index(drop=True)
df_final_gold["alerta_next_day"] = df_final_gold["alerta_next_day"].astype(int)
daily_parquet = PROCESSED_DIR / f"daily_modeling_gold_{OUTPUT_TAG}.parquet"
df_final_gold.to_parquet(daily_parquet, index=False, engine="pyarrow")

logger.info(f"Parquet Diario Gold: {daily_parquet}")
logger.info(f"Dimensiones Capa Gold: {df_final_gold.shape}")

# Resumen de positivos por ciudad y calidad del pronóstico
resumen = (
    df_final_gold.groupby("city")["alerta_next_day"]
    .agg(dias="count", alertas="sum", tasa="mean")
    .round(3)
)
logger.info("Resumen target por ciudad:\n" + resumen.to_string())

df_quality = pd.DataFrame(forecast_quality)
quality_csv = PROCESSED_DIR / f"forecast_quality_{OUTPUT_TAG}.csv"
df_quality.to_csv(quality_csv, index=False)
logger.info("Calidad del pronóstico vs ERA5:\n" + df_quality.to_string(index=False))

2026-10-18 15:36:17,390 [INFO] Directorio base detectado: C:\Users\chest\OneDrive\Documentos\Portafolio\Medicion contaminantes
2026-10-18 15:36:17,390 [INFO] Ventana analítica: 2020-01-01 a 2026-10-14
2026-10-18 15:36:17,398 [INFO] Versión de salida: prueba_final
2026-10-18 15:36:17,423 [INFO] =================================================================
2026-10-18 15:36:17,424 [INFO] Procesando: Coyhaique / Aysén (Aysén) | Zona Horaria: America/Aysen_Dynamic
2026-10-18 15:36:17,701 [INFO] [m25_aysen.csv] Horas '0' corregidas como cierre de día: 0
2026-10-18 15:36:18,514 [INFO] [aysen] Leyendo meteorología desde caché (weather_aysen_2020-01-01_2026-10-14.parquet).
2026-10-18 15:36:18,534 [INFO] [aysen] Leyendo pronóstico GFS desde caché (forecast_gfs_aysen_2020-01-01_2026-10-14.parquet).
2026-10-18 15:36:18,578 [INFO] [Aysén] Error medio pronóstico vs ERA5: temp=1.57 °C, viento=1.85 km/h
2026-10-18 15:36:18,698 [INFO] [Aysén] Capa Gold: 2,480 días | target válido: 2,435 | alertas (

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = Path("data/processed")
gold_path = PROCESSED_DIR / "daily_modeling_gold_prueba_final.parquet"
if not gold_path.exists():
  gold_path = Path("../data/processed") / "daily_modeling_gold_prueba_final.parquet"

df = pd.read_parquet(gold_path)
df["date_local"] = pd.to_datetime(df["date_local"])

FEATURES_LIN = [
    "temp_min_day_forecast",
    "wind_mean_day_forecast",
    "temperature_2m_noche_forecast",
    "wind_speed_10m_noche_forecast",
    "delta_temp_noche",
    "delta_wind_noche",
    "estancamiento_manana",
]

CONFIG = {
    "Santiago": {"desde": None, "fecha_test": "2025-01-01", "target": 50.0},
    "Temuco": {"desde": None, "fecha_test": "2025-01-01", "target": 50.0},
    "Aysén": {"desde": None, "fecha_test": "2025-01-01", "target": 80.0},
    "Valdivia": {
        "desde": "2025-04-01",
        "fecha_test": "2026-01-01",
        "target": 50.0,
    },
}

print("=" * 70)
print("  RENDIMIENTO FUERA DE MUESTRA — DELTA GFS HOMOGÉNEO (PRUEBA FINAL)")
print("=" * 70)

filas = []
for ciudad, cfg in CONFIG.items():
  d = df[(df["city"] == ciudad) & df["month"].isin([5, 6, 7, 8])].copy()
  if cfg["desde"]:
    d = d[d["date_local"] >= cfg["desde"]]

  d["y"] = (d["pm25_next_day"] >= cfg["target"]).astype(int)
  d = d.dropna(subset=FEATURES_LIN + ["y"]).reset_index(drop=True)

  tr = d[d["date_local"] < cfg["fecha_test"]]
  te = d[d["date_local"] >= cfg["fecha_test"]]

  clf = make_pipeline(
      StandardScaler(),
      LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000),
  ).fit(tr[FEATURES_LIN], tr["y"])

  probs_tr = clf.predict_proba(tr[FEATURES_LIN])[:, 1]
  probs_te = clf.predict_proba(te[FEATURES_LIN])[:, 1]

  umbral_30 = float(np.quantile(probs_tr, 0.70))
  alertas_emitidas = (probs_te >= umbral_30).mean() * 100

  auc_te = roc_auc_score(te["y"], probs_te)

  filas.append({
      "Ciudad": ciudad,
      "Target": f"≥{int(cfg['target'])} µg/m³",
      "Días Test": len(te),
      "AUC Test Ciego": round(auc_te, 3),
      "% Días Alertados": f"{alertas_emitidas:.1f}%",
  })

print(pd.DataFrame(filas).to_string(index=False))
print("=" * 70)

  RENDIMIENTO FUERA DE MUESTRA — DELTA GFS HOMOGÉNEO (PRUEBA FINAL)
  Ciudad    Target  Días Test  AUC Test Ciego % Días Alertados
Santiago ≥50 µg/m³        241           0.617            29.9%
  Temuco ≥50 µg/m³        230           0.851            26.5%
   Aysén ≥80 µg/m³        246           0.883            35.8%
Valdivia ≥50 µg/m³        123           0.746            26.0%


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = Path("data/processed")
gold_path = PROCESSED_DIR / "daily_modeling_gold_prueba_final.parquet"
if not gold_path.exists():
  gold_path = Path("../data/processed") / "daily_modeling_gold_prueba_final.parquet"

df = pd.read_parquet(gold_path)
df["date_local"] = pd.to_datetime(df["date_local"])

FEATURES_LIN = [
    "temp_min_day_forecast",
    "wind_mean_day_forecast",
    "temperature_2m_noche_forecast",
    "wind_speed_10m_noche_forecast",
    "delta_temp_noche",
    "delta_wind_noche",
    "estancamiento_manana",
]

CONFIG = {
    "Santiago": {"desde": None, "fecha_test": "2025-01-01", "target": 50.0},
    "Temuco": {"desde": None, "fecha_test": "2025-01-01", "target": 50.0},
    "Aysén": {"desde": None, "fecha_test": "2025-01-01", "target": 80.0},
    "Valdivia": {
        "desde": "2025-04-01",
        "fecha_test": "2026-01-01",
        "target": 50.0,
    },
}

res_detalle = []

for ciudad, cfg in CONFIG.items():
  d = df[(df["city"] == ciudad) & df["month"].isin([5, 6, 7, 8])].copy()
  if cfg["desde"]:
    d = d[d["date_local"] >= cfg["desde"]]

  d["y"] = (d["pm25_next_day"] >= cfg["target"]).astype(int)
  d["alerta_hoy"] = (d["pm25_mean_day"] >= cfg["target"]).astype(int)
  d = d.dropna(subset=FEATURES_LIN + ["y", "alerta_hoy"]).reset_index(drop=True)

  tr = d[d["date_local"] < cfg["fecha_test"]]
  te = d[d["date_local"] >= cfg["fecha_test"]].copy()

  clf = make_pipeline(
      StandardScaler(),
      LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000),
  ).fit(tr[FEATURES_LIN], tr["y"])

  probs_tr = clf.predict_proba(tr[FEATURES_LIN])[:, 1]
  probs_te = clf.predict_proba(te[FEATURES_LIN])[:, 1]

  umbral_30 = float(np.quantile(probs_tr, 0.70))
  pred = (probs_te >= umbral_30).astype(int)
  y_true = te["y"].to_numpy()

  # Matriz de confusión
  tp = int(((pred == 1) & (y_true == 1)).sum())  # Aciertos reales de alerta
  fp = int(((pred == 1) & (y_true == 0)).sum())  # Falsas alarmas
  fn = int(((pred == 0) & (y_true == 1)).sum())  # Alertas perdidas
  tn = int(((pred == 0) & (y_true == 0)).sum())  # Días limpios bien predichos

  # Primer día (inicio de episodio: hoy limpio, mañana alerta)
  ini_mask = (te["alerta_hoy"] == 0) & (te["y"] == 1)
  n_ini = int(ini_mask.sum())
  tp_ini = int(pred[ini_mask].sum())

  res_detalle.append({
      "Ciudad": ciudad,
      "Días Totales": len(te),
      "Alertas Reales": int(y_true.sum()),
      "Alertas Emitidas": int(pred.sum()),
      "Aciertos (TP)": tp,
      "Falsas Alarmas (FP)": fp,
      "Alertas Perdidas (FN)": fn,
      "Precisión": f"{tp / (tp + fp):.1%}" if (tp + fp) > 0 else "N/A",
      "Recall Total": f"{tp / (tp + fn):.1%}",
      "1er Día Detectado": f"{tp_ini}/{n_ini} ({tp_ini / n_ini:.1%})"
      if n_ini > 0
      else "N/A",
  })

print("=" * 110)
print(
    "  DESGLOSE OPERATIVO DE ACIERTOS Y FALLOS — TEST CIEGO (PRESUPUESTO 30%)"
)
print("=" * 110)
print(pd.DataFrame(res_detalle).to_string(index=False))
print("=" * 110)

  DESGLOSE OPERATIVO DE ACIERTOS Y FALLOS — TEST CIEGO (PRESUPUESTO 30%)
  Ciudad  Días Totales  Alertas Reales  Alertas Emitidas  Aciertos (TP)  Falsas Alarmas (FP)  Alertas Perdidas (FN) Precisión Recall Total 1er Día Detectado
Santiago           241              42                72             16                   56                     26     22.2%        38.1%      5/16 (31.2%)
  Temuco           230              84                61             47                   14                     37     77.0%        56.0%     16/29 (55.2%)
   Aysén           246              96                88             71                   17                     25     80.7%        74.0%     18/33 (54.5%)
Valdivia           123              39                32             19                   13                     20     59.4%        48.7%      9/17 (52.9%)


In [5]:
"""
03_train_production_models.py
Reentrena los 4 modelos finales de producción utilizando la serie histórica
completa (dataset Gold homogéneo GFS-GFS) y fija el presupuesto operativo al 30%.
"""
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Definición de rutas
CURRENT_DIR = Path().resolve()
if (CURRENT_DIR / "data" / "processed").exists():
    BASE_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "data" / "processed").exists():
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Carga del dataset Gold homogéneo (prueba_final)
gold_path = PROCESSED_DIR / "daily_modeling_gold_prueba_final.parquet"
if not gold_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo Gold en: {gold_path}")

df_gold = pd.read_parquet(gold_path)
df_gold["date_local"] = pd.to_datetime(df_gold["date_local"])

# 3. Configuración del entrenamiento
TAG = "v2"
MESES_GEC = [5, 6, 7, 8]  # Ventana invernal
BUDGET_NOMINAL = 0.30     # Presupuesto parejo del 30% de días de mayor riesgo
UMBRAL_TARGET_DEFECTO = 50.0

CONFIG_CIUDADES = {
    "Santiago": {"desde": None, "target": 50.0},
    "Temuco":   {"desde": None, "target": 50.0},
    "Aysén":    {"desde": None, "target": 80.0},
    "Valdivia": {"desde": "2025-04-01", "target": 50.0},
}

FEATURES_LIN = [
    "temp_min_day_forecast",
    "wind_mean_day_forecast",
    "temperature_2m_noche_forecast",
    "wind_speed_10m_noche_forecast",
    "delta_temp_noche",
    "delta_wind_noche",
    "estancamiento_manana",
]

def make_logit():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000)
    )

print("=" * 80)
print("  REENTRENAMIENTO Y SERIALIZACIÓN DE MODELOS DE PRODUCCIÓN (SERIE COMPLETA)")
print("=" * 80)

for ciudad, cfg in CONFIG_CIUDADES.items():
    # Filtrado por cuenca y meses invernales
    d = df_gold[(df_gold["city"] == ciudad) & df_gold["month"].isin(MESES_GEC)].copy()
    if cfg["desde"]:
        d = d[d["date_local"] >= cfg["desde"]]
    
    target_val = cfg["target"]
    d["y"] = (d["pm25_next_day"] >= target_val).astype(int)
    d = d.dropna(subset=FEATURES_LIN + ["y"]).reset_index(drop=True)
    
    # Ajuste del modelo sobre el 100% de la historia disponible
    clf = make_logit().fit(d[FEATURES_LIN], d["y"])
    
    # Calibración del corte operativo al 30% superior de probabilidades
    probs = clf.predict_proba(d[FEATURES_LIN])[:, 1]
    decision_threshold = float(np.quantile(probs, 1.0 - BUDGET_NOMINAL))
    
    # Empaquetado del artefacto con metadatos completos y llaves redundantes
    payload = {
        "ciudad": ciudad,
        "tipo_estrategia": "modelo_puro_meteorologico_gfs",
        "modelo": clf,
        "model": clf,
        "features": FEATURES_LIN,
        "decision_threshold": decision_threshold,
        "umbral": decision_threshold,
        "target_umbral_pm25": target_val,
        "budget_nominal": BUDGET_NOMINAL,
        "training_window": f"{d['date_local'].min().date()} a {d['date_local'].max().date()}",
        "n_training_days": len(d),
        "tag_version": TAG,
    }
    
    slug = "aysen" if ciudad.lower() in ["aysén", "aysen"] else ciudad.lower()
    filename = f"alert_model_{slug}_{TAG}.joblib"
    filepath = MODELS_DIR / filename
    
    joblib.dump(payload, filepath)
    
    print(f"\n[{ciudad.upper()}]")
    print(f"  • Archivo guardado : {filepath.name}")
    print(f"  • Ventana temporal : {payload['training_window']} ({payload['n_training_days']} días)")
    print(f"  • Target evaluado  : ≥ {int(target_val)} µg/m³")
    print(f"  • Umbral operativo : {decision_threshold:.3f} (Top {int(BUDGET_NOMINAL*100)}% de riesgo)")

print("\n" + "=" * 80)
print("✓ Artefactos actualizados exitosamente en la carpeta models/")
print("=" * 80)

  REENTRENAMIENTO Y SERIALIZACIÓN DE MODELOS DE PRODUCCIÓN (SERIE COMPLETA)

[SANTIAGO]
  • Archivo guardado : alert_model_santiago_v2.joblib
  • Ventana temporal : 2021-05-01 a 2026-08-31 (719 días)
  • Target evaluado  : ≥ 50 µg/m³
  • Umbral operativo : 0.583 (Top 30% de riesgo)

[TEMUCO]
  • Archivo guardado : alert_model_temuco_v2.joblib
  • Ventana temporal : 2021-05-01 a 2026-08-31 (717 días)
  • Target evaluado  : ≥ 50 µg/m³
  • Umbral operativo : 0.688 (Top 30% de riesgo)

[AYSÉN]
  • Archivo guardado : alert_model_aysen_v2.joblib
  • Ventana temporal : 2021-05-01 a 2026-08-31 (736 días)
  • Target evaluado  : ≥ 80 µg/m³
  • Umbral operativo : 0.692 (Top 30% de riesgo)

[VALDIVIA]
  • Archivo guardado : alert_model_valdivia_v2.joblib
  • Ventana temporal : 2025-05-01 a 2026-08-31 (246 días)
  • Target evaluado  : ≥ 50 µg/m³
  • Umbral operativo : 0.622 (Top 30% de riesgo)

✓ Artefactos actualizados exitosamente en la carpeta models/


In [6]:
# ==============================================================================
# CELDA 8: SERIALIZACIÓN DE MODELOS DE PRODUCCIÓN (GOLD: prueba_final)
# ==============================================================================
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Definición robusta de rutas
CURRENT_DIR = Path().resolve()
if (CURRENT_DIR / "data" / "processed").exists():
  BASE_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "data" / "processed").exists():
  BASE_DIR = CURRENT_DIR.parent
else:
  BASE_DIR = CURRENT_DIR

PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Cargar el dataset Gold generado por el ETL final
GOLD_FILENAME = "daily_modeling_gold_prueba_final.parquet"
gold_path = PROCESSED_DIR / GOLD_FILENAME

if not gold_path.exists():
  raise FileNotFoundError(
      f"No se encontró el archivo Gold en: {gold_path}\n"
      "Asegúrate de haber ejecutado el nuevo ETL con OUTPUT_TAG = 'prueba_final'."
  )

df_gold = pd.read_parquet(gold_path)
df_gold["date_local"] = pd.to_datetime(df_gold["date_local"])

# 3. Parámetros del modelo de producción
TAG_MODELO = "v2"  # Mantenemos v2 como versión oficial de los artefactos
MESES_GEC = [5, 6, 7, 8]  # Ventana invernal
BUDGET_NOMINAL = 0.30  # Presupuesto parejo de alertas (30% de mayor riesgo)
UMBRAL_TARGET_DEFECTO = 50.0

CONFIG_CIUDADES = {
    "Santiago": {"desde": None, "target": 50.0},
    "Temuco": {"desde": None, "target": 50.0},
    "Aysén": {"desde": None, "target": 80.0},  # Norma y dinámica local
    "Valdivia": {
        "desde": "2025-04-01",
        "target": 50.0,
    },  # Serie post-quiebre instrumental
}

# Features físicas homogéneas (GFS vs GFS)
FEATURES_LIN = [
    "temp_min_day_forecast",
    "wind_mean_day_forecast",
    "temperature_2m_noche_forecast",
    "wind_speed_10m_noche_forecast",
    "delta_temp_noche",
    "delta_wind_noche",
    "estancamiento_manana",
]


def make_logit():
  return make_pipeline(
      StandardScaler(),
      LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000),
  )


print("=" * 80)
print(f"  SERIALIZACIÓN DE MODELOS DE PRODUCCIÓN (FUENTE: {GOLD_FILENAME})")
print("=" * 80)

for ciudad, cfg in CONFIG_CIUDADES.items():
  # Filtrar por cuenca, periodo válido y meses invernales
  d = df_gold[
      (df_gold["city"] == ciudad) & df_gold["month"].isin(MESES_GEC)
  ].copy()
  if cfg["desde"]:
    d = d[d["date_local"] >= cfg["desde"]]

  target_val = cfg["target"]
  d["y"] = (d["pm25_next_day"] >= target_val).astype(int)
  d = d.dropna(subset=FEATURES_LIN + ["y"]).reset_index(drop=True)

  # Ajuste sobre el 100% de la serie histórica disponible
  clf = make_logit().fit(d[FEATURES_LIN], d["y"])

  # Calibración del corte de probabilidad al percentil 70 (top 30% de riesgo)
  probs = clf.predict_proba(d[FEATURES_LIN])[:, 1]
  decision_threshold = float(np.quantile(probs, 1.0 - BUDGET_NOMINAL))

  # Empaquetado estandarizado
  payload = {
      "ciudad": ciudad,
      "tipo_estrategia": "modelo_puro_meteorologico_gfs",
      "modelo": clf,
      "model": clf,  # Doble llave para evitar KeyErrors
      "features": FEATURES_LIN,
      "decision_threshold": decision_threshold,
      "umbral": decision_threshold,  # Doble llave para evitar KeyErrors
      "target_umbral_pm25": target_val,
      "budget_nominal": BUDGET_NOMINAL,
      "training_window": (
          f"{d['date_local'].min().date()} a {d['date_local'].max().date()}"
      ),
      "n_training_days": len(d),
      "gold_source": GOLD_FILENAME,
      "tag_version": TAG_MODELO,
  }

  slug = "aysen" if ciudad.lower() in ["aysén", "aysen"] else ciudad.lower()
  filename = f"alert_model_{slug}_{TAG_MODELO}.joblib"
  filepath = MODELS_DIR / filename

  joblib.dump(payload, filepath)

  print(f"\n[{ciudad.upper()}]")
  print(f"  • Archivo guardado : {filepath.name}")
  print(
      f"  • Rango histórico  : {payload['training_window']} ({payload['n_training_days']} días invernales)"
  )
  print(f"  • Target evaluado  : ≥ {int(target_val)} µg/m³")
  print(
      f"  • Umbral operativo : {decision_threshold:.3f} (Top {int(BUDGET_NOMINAL*100)}% de riesgo)"
  )

print("\n" + "=" * 80)
print(f"✓ Modelos sincronizados exitosamente en: {MODELS_DIR}")
print("=" * 80)

  SERIALIZACIÓN DE MODELOS DE PRODUCCIÓN (FUENTE: daily_modeling_gold_prueba_final.parquet)

[SANTIAGO]
  • Archivo guardado : alert_model_santiago_v2.joblib
  • Rango histórico  : 2021-05-01 a 2026-08-31 (719 días invernales)
  • Target evaluado  : ≥ 50 µg/m³
  • Umbral operativo : 0.583 (Top 30% de riesgo)

[TEMUCO]
  • Archivo guardado : alert_model_temuco_v2.joblib
  • Rango histórico  : 2021-05-01 a 2026-08-31 (717 días invernales)
  • Target evaluado  : ≥ 50 µg/m³
  • Umbral operativo : 0.688 (Top 30% de riesgo)

[AYSÉN]
  • Archivo guardado : alert_model_aysen_v2.joblib
  • Rango histórico  : 2021-05-01 a 2026-08-31 (736 días invernales)
  • Target evaluado  : ≥ 80 µg/m³
  • Umbral operativo : 0.692 (Top 30% de riesgo)

[VALDIVIA]
  • Archivo guardado : alert_model_valdivia_v2.joblib
  • Rango histórico  : 2025-05-01 a 2026-08-31 (246 días invernales)
  • Target evaluado  : ≥ 50 µg/m³
  • Umbral operativo : 0.622 (Top 30% de riesgo)

✓ Modelos sincronizados exitosamente en: C:\U

In [7]:
"""src/04_predict_tomorrow.py

Inferencia operativa diaria: pronostica el riesgo de alerta PM2.5 para mañana
(t+1)
consumiendo los modelos entrenados con la formulación homogénea GFS-GFS.
"""

from datetime import datetime
import logging
from pathlib import Path
from zoneinfo import ZoneInfo
import joblib
import numpy as np
import pandas as pd
import requests
import yaml

# ------------------------------------------------------------------------------
# 1. CONFIGURACIÓN Y RUTAS
# ------------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("InferenciaOperativa")

BASE_DIR = (
    Path(__file__).resolve().parent.parent
    if "__file__" in locals()
    else Path().resolve()
)
CONFIG_PATH = BASE_DIR / "stations.yaml"
MODELS_DIR = BASE_DIR / "models"
PREDICTIONS_DIR = BASE_DIR / "data" / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
HISTORIAL_CSV = PREDICTIONS_DIR / "historial_predicciones.csv"

TAG_MODELO = "v2"

FEATURES_LIN = [
    "temp_min_day_forecast",
    "wind_mean_day_forecast",
    "temperature_2m_noche_forecast",
    "wind_speed_10m_noche_forecast",
    "delta_temp_noche",
    "delta_wind_noche",
    "estancamiento_manana",
]

if not CONFIG_PATH.exists():
  raise FileNotFoundError(f"No se encontró stations.yaml en: {CONFIG_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
  stations_config = yaml.safe_load(f)["stations"]

now_utc = datetime.now(ZoneInfo("UTC"))
resultados = []

print("=" * 100)
print("  SISTEMA OPERATIVO DE ALERTA TEMPRANA PM2.5 — INFERENCIA DIARIA (GFS)")
print("=" * 100)

# ------------------------------------------------------------------------------
# 2. INFERENCIA POR CUENCA
# ------------------------------------------------------------------------------
for st in stations_config:
  ciudad = st["city"]
  lat = st["latitude"]
  lon = st["longitude"]
  tz_str = st.get("timezone", "America/Santiago")

  tz_local = (
      ZoneInfo("America/Punta_Arenas")
      if tz_str == "America/Aysen_Dynamic"
      else ZoneInfo(tz_str)
  )

  local_now = now_utc.astimezone(tz_local)
  fecha_hoy = local_now.date()
  fecha_manana = fecha_hoy + pd.Timedelta(days=1)

  slug_ciudad = (
      "aysen" if ciudad.lower() in ["aysén", "aysen"] else ciudad.lower()
  )
  modelo_path = MODELS_DIR / f"alert_model_{slug_ciudad}_{TAG_MODELO}.joblib"

  if not modelo_path.exists():
    logger.error(
        f"[{ciudad}] No se encontró el artefacto: {modelo_path.name}. Se"
        " omite."
    )
    continue

  artefacto = joblib.load(modelo_path)
  modelo = (
      artefacto.get("modelo")
      if artefacto.get("modelo") is not None
      else artefacto.get("model")
  )

  umbral = artefacto.get("decision_threshold")
  if umbral is None:
    umbral = artefacto.get("umbral", 0.5)

  target_pm25 = artefacto.get("target_umbral_pm25", 50.0)

  # Consulta a Open-Meteo GFS con past_days=1 para asegurar la noche de hoy completa
  url = "https://api.open-meteo.com/v1/forecast"
  params = {
      "latitude": lat,
      "longitude": lon,
      "hourly": ["temperature_2m", "wind_speed_10m"],
      "models": "gfs_seamless",
      "past_days": 1,
      "forecast_days": 3,
      "timezone": "UTC",
  }

  try:
    resp = requests.get(url, params=params, timeout=25)
    resp.raise_for_status()
    payload = resp.json()["hourly"]
  except Exception as e:
    logger.error(f"[{ciudad}] Error al consultar Open-Meteo: {e}")
    continue

  df_fc = pd.DataFrame(payload)
  df_fc["time_utc"] = pd.to_datetime(df_fc["time"]).dt.tz_localize("UTC")
  df_fc["time_local"] = df_fc["time_utc"].dt.tz_convert(tz_local)
  df_fc["date_local"] = df_fc["time_local"].dt.date
  df_fc["hour_local"] = df_fc["time_local"].dt.hour

  # A. Noche de hoy (18:00 a 23:00)
  noche_hoy = df_fc[
      (df_fc["date_local"] == fecha_hoy)
      & (df_fc["hour_local"].between(18, 23))
  ]
  if len(noche_hoy) < 5:
    logger.error(
        f"[{ciudad}] Noche de hoy incompleta ({len(noche_hoy)}/6 horas)."
        " Se rechaza inferencia para evitar cálculo de delta inválido."
    )
    continue

  temp_noche_hoy = noche_hoy["temperature_2m"].mean()
  wind_noche_hoy = noche_hoy["wind_speed_10m"].mean()

  # B. Día de mañana (t+1)
  dia_manana = df_fc[df_fc["date_local"] == fecha_manana]
  if len(dia_manana) < 20:
    logger.error(
        f"[{ciudad}] Pronóstico de mañana incompleto ({len(dia_manana)}/24"
        " horas). Se omite."
    )
    continue

  temp_min_day_fc = dia_manana["temperature_2m"].min()
  wind_mean_day_fc = dia_manana["wind_speed_10m"].mean()

  # C. Noche de mañana (18:00 a 23:00 de t+1)
  noche_manana = dia_manana[dia_manana["hour_local"].between(18, 23)]
  temp_noche_manana = noche_manana["temperature_2m"].mean()
  wind_noche_manana = noche_manana["wind_speed_10m"].mean()

  # D. Deltas GFS vs GFS
  delta_temp = temp_noche_manana - temp_noche_hoy
  delta_wind = wind_noche_manana - wind_noche_hoy
  estancamiento = wind_noche_manana / np.maximum(temp_noche_manana + 5.0, 1.0)

  X_pred = pd.DataFrame([{
      "temp_min_day_forecast": temp_min_day_fc,
      "wind_mean_day_forecast": wind_mean_day_fc,
      "temperature_2m_noche_forecast": temp_noche_manana,
      "wind_speed_10m_noche_forecast": wind_noche_manana,
      "delta_temp_noche": delta_temp,
      "delta_wind_noche": delta_wind,
      "estancamiento_manana": estancamiento,
  }])[FEATURES_LIN]

  probabilidad = float(modelo.predict_proba(X_pred)[:, 1][0])
  alerta_emitida = int(probabilidad >= umbral)

  resultados.append({
      "fecha_ejecucion": local_now.strftime("%Y-%m-%d %H:%M"),
      "ciudad": ciudad,
      "fecha_predicha": str(fecha_manana),
      "target_umbral": f"≥{int(target_pm25)} µg/m³",
      "probabilidad": round(probabilidad, 3),
      "umbral_operativo": round(umbral, 3),
      "alerta_emitida": "SÍ" if alerta_emitida == 1 else "NO",
      "temp_min_manana_C": round(temp_min_day_fc, 1),
      "viento_medio_manana_kmh": round(wind_mean_day_fc, 1),
      "delta_temp_noche_C": round(delta_temp, 2),
      "delta_viento_noche_kmh": round(delta_wind, 2),
      "modelo_usado": modelo_path.name,
  })

# ------------------------------------------------------------------------------
# 3. REPORTE EN CONSOLA Y PERSISTENCIA
# ------------------------------------------------------------------------------
df_predicciones = pd.DataFrame(resultados)

if not df_predicciones.empty:
  cols_vista = [
      "ciudad",
      "fecha_predicha",
      "target_umbral",
      "probabilidad",
      "umbral_operativo",
      "alerta_emitida",
      "temp_min_manana_C",
      "viento_medio_manana_kmh",
      "delta_temp_noche_C",
  ]
  print(df_predicciones[cols_vista].to_string(index=False))
  print("=" * 100)

  if HISTORIAL_CSV.exists():
    df_previo = pd.read_csv(HISTORIAL_CSV)
    df_acum = pd.concat([df_previo, df_predicciones], ignore_index=True)
    df_acum = df_acum.drop_duplicates(
        subset=["ciudad", "fecha_predicha"], keep="last"
    )
    df_acum.to_csv(HISTORIAL_CSV, index=False)
  else:
    df_predicciones.to_csv(HISTORIAL_CSV, index=False)

  logger.info(f"Historial operativo guardado en: {HISTORIAL_CSV}")
else:
  print("No se generaron predicciones.")
  print("=" * 100)

  SISTEMA OPERATIVO DE ALERTA TEMPRANA PM2.5 — INFERENCIA DIARIA (GFS)


2026-10-18 15:54:47,260 [INFO] Historial operativo guardado en: C:\Users\chest\OneDrive\Documentos\Portafolio\Medicion contaminantes\data\predictions\historial_predicciones.csv


  ciudad fecha_predicha target_umbral  probabilidad  umbral_operativo alerta_emitida  temp_min_manana_C  viento_medio_manana_kmh  delta_temp_noche_C
   Aysén     2026-10-19     ≥80 µg/m³         0.309             0.692             NO                6.5                      3.3                1.25
Valdivia     2026-10-19     ≥50 µg/m³         0.269             0.622             NO                9.5                      3.5               -2.00
  Temuco     2026-10-19     ≥50 µg/m³         0.132             0.688             NO                9.7                      4.7               -2.57
Santiago     2026-10-19     ≥50 µg/m³         0.186             0.583             NO               14.6                      6.9                0.52


In [8]:
"""
src/05_evaluate_live_performance.py
Auditoría y evaluación de desempeño operativo en vivo.
Cruza el historial acumulado de predicciones con las observaciones consolidadas de SINCA.
"""
from pathlib import Path
import logging
import pandas as pd
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("LiveEvaluation")

# 1. Definición de rutas
BASE_DIR = Path(__file__).resolve().parent.parent if "__file__" in locals() else Path().resolve()
PREDICTIONS_CSV = BASE_DIR / "data" / "predictions" / "historial_predicciones.csv"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTE_CSV = BASE_DIR / "data" / "predictions" / "reporte_desempeno_live.csv"

if not PREDICTIONS_CSV.exists():
    raise FileNotFoundError(f"No existe el archivo de historial: {PREDICTIONS_CSV}")

# 2. Cargar historial de predicciones
df_pred = pd.read_csv(PREDICTIONS_CSV)
df_pred["fecha_predicha"] = pd.to_datetime(df_pred["fecha_predicha"]).dt.date

# 3. Cargar datos observados (Parquet Gold o histórico procesado)
# Se busca el archivo gold disponible para extraer el promedio diario observado real
gold_files = list(PROCESSED_DIR.glob("daily_modeling_gold_*.parquet"))
if not gold_files:
    raise FileNotFoundError(f"No se encontraron datasets procesados en: {PROCESSED_DIR}")

# Seleccionar el archivo gold más reciente
gold_path = sorted(gold_files)[-1]
logger.info(f"Cargando observaciones reales desde: {gold_path.name}")
df_obs = pd.read_parquet(gold_path)
df_obs["date_local"] = pd.to_datetime(df_obs["date_local"]).dt.date

# Filtrar solo columnas relevantes de observación
df_obs_clean = df_obs[["city", "date_local", "pm25_mean_day"]].dropna(subset=["pm25_mean_day"])

# 4. Cruce de predicción vs realidad
merged = df_pred.merge(
    df_obs_clean,
    left_on=["ciudad", "fecha_predicha"],
    right_on=["city", "date_local"],
    how="inner"
)

print("=" * 95)
print("  EVALUACIÓN DE DESEMPEÑO OPERATIVO EN VIVO (PREDICCIONES VS SINCA OBSERVADO)")
print("=" * 95)

if merged.empty:
    print("\nAún no hay fechas predichas consolidadas en el dataset de observaciones.")
    print("A medida que transcurran los días y se actualice la capa de datos de SINCA,")
    print("este script evaluará automáticamente los aciertos y falsas alarmas.")
    print(f"Total predicciones registradas pendientes de evaluación: {len(df_pred)}")
    print("=" * 95)
else:
    # Determinar el target numérico a partir del texto (ej. "≥50 µg/m³" -> 50.0)
    merged["target_num"] = merged["target_umbral"].str.extract(r"(\d+)").astype(float)
    merged["y_real"] = (merged["pm25_mean_day"] >= merged["target_num"]).astype(int)
    merged["pred_binaria"] = (merged["alerta_emitida"] == "SÍ").astype(int)

    resumen = []
    for ciudad, d in merged.groupby("ciudad"):
        tp = int(((d["pred_binaria"] == 1) & (d["y_real"] == 1)).sum())
        fp = int(((d["pred_binaria"] == 1) & (d["y_real"] == 0)).sum())
        fn = int(((d["pred_binaria"] == 0) & (d["y_real"] == 1)).sum())
        tn = int(((d["pred_binaria"] == 0) & (d["y_real"] == 0)).sum())
        
        total = len(d)
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan

        resumen.append({
            "Ciudad": ciudad,
            "Días Evaluados": total,
            "Alertas Reales": int(d["y_real"].sum()),
            "Alertas Emitidas": int(d["pred_binaria"].sum()),
            "Aciertos (TP)": tp,
            "Falsas Alarmas (FP)": fp,
            "Alertas Perdidas (FN)": fn,
            "Precisión": f"{precision:.1%}" if not np.isnan(precision) else "N/A",
            "Recall": f"{recall:.1%}" if not np.isnan(recall) else "N/A"
        })

    df_resumen = pd.DataFrame(resumen)
    print(df_resumen.to_string(index=False))
    print("=" * 95)
    
    merged.to_csv(REPORTE_CSV, index=False)
    logger.info(f"Detalle de auditoría exportado a: {REPORTE_CSV}")

2026-10-18 15:56:03,652 [INFO] Cargando observaciones reales desde: daily_modeling_gold_v2.parquet


  EVALUACIÓN DE DESEMPEÑO OPERATIVO EN VIVO (PREDICCIONES VS SINCA OBSERVADO)

Aún no hay fechas predichas consolidadas en el dataset de observaciones.
A medida que transcurran los días y se actualice la capa de datos de SINCA,
este script evaluará automáticamente los aciertos y falsas alarmas.
Total predicciones registradas pendientes de evaluación: 4


In [9]:
"""
02_evaluate_benchmark_final.py
Evaluación científica estricta fuera de muestra (Test ciego 2025-2026).
Calcula métricas operativas al 30% de presupuesto e intervalos de confianza (IC 95%)
mediante bootstrap de bloques semanales.
"""
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# 1. Cargar capa Gold homogénea nueva
CURRENT_DIR = Path().resolve()
BASE_DIR = CURRENT_DIR if (CURRENT_DIR / "data" / "processed").exists() else CURRENT_DIR.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
gold_path = PROCESSED_DIR / "daily_modeling_gold_prueba_final.parquet"

df = pd.read_parquet(gold_path)
df["date_local"] = pd.to_datetime(df["date_local"])

FEATURES_LIN = [
    "temp_min_day_forecast",
    "wind_mean_day_forecast",
    "temperature_2m_noche_forecast",
    "wind_speed_10m_noche_forecast",
    "delta_temp_noche",
    "delta_wind_noche",
    "estancamiento_manana",
]

CONFIG = {
    "Santiago": {"desde": None, "fecha_test": "2025-01-01", "target": 50.0},
    "Temuco":   {"desde": None, "fecha_test": "2025-01-01", "target": 50.0},
    "Aysén":    {"desde": None, "fecha_test": "2025-01-01", "target": 80.0},
    "Valdivia": {"desde": "2025-04-01", "fecha_test": "2026-01-01", "target": 50.0},
}

def bootstrap_metricas(te_df, n_boot=500, seed=42):
    """Calcula IC 95% para Recall 1er día, Precisión y AUC respetando bloques semanales."""
    te = te_df.reset_index(drop=True)
    y = te["y"].to_numpy()
    f = te["pred"].to_numpy()
    p = te["prob"].to_numpy()
    ini = ((te["alerta_hoy"] == 0) & (te["y"] == 1)).to_numpy()
    
    grupos = te.groupby(te["date_local"].dt.to_period("W").astype(str)).indices
    claves = list(grupos)
    rng = np.random.default_rng(seed)
    
    aucs, recs_ini, precs = [], [], []
    for _ in range(n_boot):
        elegidos = rng.choice(len(claves), size=len(claves), replace=True)
        idx = np.concatenate([grupos[claves[i]] for i in elegidos])
        yi, fi, pi, inii = y[idx], f[idx], p[idx], ini[idx]
        
        if len(np.unique(yi)) > 1:
            aucs.append(roc_auc_score(yi, pi))
        if inii.sum() > 0:
            recs_ini.append(fi[inii].mean())
        if fi.sum() > 0:
            precs.append(((fi == 1) & (yi == 1)).sum() / fi.sum())
            
    fmt = lambda v: f"[{np.percentile(v, 2.5):.2f} - {np.percentile(v, 97.5):.2f}]" if len(v) >= 50 else "N/A"
    return fmt(aucs), fmt(recs_ini), fmt(precs)

filas = []

for ciudad, cfg in CONFIG.items():
    d = df[(df["city"] == ciudad) & df["month"].isin([5, 6, 7, 8])].copy()
    if cfg["desde"]:
        d = d[d["date_local"] >= cfg["desde"]]
        
    d["y"] = (d["pm25_next_day"] >= cfg["target"]).astype(int)
    d["alerta_hoy"] = (d["pm25_mean_day"] >= cfg["target"]).astype(int)
    d = d.dropna(subset=FEATURES_LIN + ["y", "alerta_hoy"]).reset_index(drop=True)
    
    tr = d[d["date_local"] < cfg["fecha_test"]].copy()
    te = d[d["date_local"] >= cfg["fecha_test"]].copy()
    
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000)
    ).fit(tr[FEATURES_LIN], tr["y"])
    
    probs_tr = clf.predict_proba(tr[FEATURES_LIN])[:, 1]
    umbral_30 = float(np.quantile(probs_tr, 0.70))
    
    te["prob"] = clf.predict_proba(te[FEATURES_LIN])[:, 1]
    te["pred"] = (te["prob"] >= umbral_30).astype(int)
    
    y_true = te["y"].to_numpy()
    pred = te["pred"].to_numpy()
    
    tp = int(((pred == 1) & (y_true == 1)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    
    ini_mask = (te["alerta_hoy"] == 0) & (te["y"] == 1)
    n_ini = int(ini_mask.sum())
    tp_ini = int(pred[ini_mask].sum())
    
    auc_val = roc_auc_score(y_true, te["prob"])
    rec_ini_val = tp_ini / n_ini if n_ini > 0 else np.nan
    prec_val = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    
    ic_auc, ic_rec_ini, ic_prec = bootstrap_metricas(te)
    
    filas.append({
        "Ciudad": ciudad,
        "Target": f"≥{int(cfg['target'])} µg/m³",
        "Días Test": len(te),
        "Alertas Reales": int(y_true.sum()),
        "AUC": f"{auc_val:.3f} {ic_auc}",
        "Precisión": f"{prec_val:.1%} {ic_prec}",
        "Recall Total": f"{tp / (tp + fn):.1%}",
        "1er Día (Recall)": f"{tp_ini}/{n_ini} ({rec_ini_val:.1%}) {ic_rec_ini}",
        "Falsas Alarmas": fp
    })

df_benchmark = pd.DataFrame(filas)
print("\n" + "=" * 125)
print("  TABLA OFICIAL DEFINITIVA — TEST CIEGO FUERA DE MUESTRA (CON INTERVALOS IC 95%)")
print("=" * 125)
print(df_benchmark.to_string(index=False))
print("=" * 125)

out_csv = PROCESSED_DIR / "benchmark_oficial_prueba_final.csv"
df_benchmark.to_csv(out_csv, index=False)
print(f"Tabla guardada en: {out_csv}")


  TABLA OFICIAL DEFINITIVA — TEST CIEGO FUERA DE MUESTRA (CON INTERVALOS IC 95%)
  Ciudad    Target  Días Test  Alertas Reales                 AUC           Precisión Recall Total            1er Día (Recall)  Falsas Alarmas
Santiago ≥50 µg/m³        241              42 0.617 [0.51 - 0.72] 22.2% [0.11 - 0.34]        38.1%  5/16 (31.2%) [0.12 - 0.48]              56
  Temuco ≥50 µg/m³        230              84 0.851 [0.78 - 0.91] 77.0% [0.64 - 0.88]        56.0% 16/29 (55.2%) [0.38 - 0.71]              14
   Aysén ≥80 µg/m³        246              96 0.883 [0.84 - 0.93] 80.7% [0.68 - 0.90]        74.0% 18/33 (54.5%) [0.37 - 0.74]              17
Valdivia ≥50 µg/m³        123              39 0.746 [0.63 - 0.85] 59.4% [0.39 - 0.76]        48.7%  9/17 (52.9%) [0.31 - 0.74]              13
Tabla guardada en: C:\Users\chest\OneDrive\Documentos\Portafolio\Medicion contaminantes\data\processed\benchmark_oficial_prueba_final.csv


In [10]:
# ==============================================================================
# EVALUACIÓN CRÍTICA: TASA BASE, LIFT SOBRE EL AZAR Y COSTO DE FALSAS ALARMAS
# ==============================================================================
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CURRENT_DIR = Path().resolve()
BASE_DIR = (
    CURRENT_DIR
    if (CURRENT_DIR / "data" / "processed").exists()
    else CURRENT_DIR.parent
)
PROCESSED_DIR = BASE_DIR / "data" / "processed"
gold_path = PROCESSED_DIR / "daily_modeling_gold_prueba_final.parquet"

df = pd.read_parquet(gold_path)
df["date_local"] = pd.to_datetime(df["date_local"])

FEATURES_LIN = [
    "temp_min_day_forecast",
    "wind_mean_day_forecast",
    "temperature_2m_noche_forecast",
    "wind_speed_10m_noche_forecast",
    "delta_temp_noche",
    "delta_wind_noche",
    "estancamiento_manana",
]

CONFIG = {
    "Santiago": {"desde": None, "fecha_test": "2025-01-01", "target": 50.0},
    "Temuco": {"desde": None, "fecha_test": "2025-01-01", "target": 50.0},
    "Aysén": {"desde": None, "fecha_test": "2025-01-01", "target": 80.0},
    "Valdivia": {
        "desde": "2025-04-01",
        "fecha_test": "2026-01-01",
        "target": 50.0,
    },
}


def bootstrap_lift(te_df, n_boot=500, seed=42):
  """Bootstrap por bloques semanales para intervalos IC 95% de Precisión y Lift."""
  te = te_df.reset_index(drop=True)
  y = te["y"].to_numpy()
  f = te["pred"].to_numpy()

  grupos = te.groupby(te["date_local"].dt.to_period("W").astype(str)).indices
  claves = list(grupos)
  rng = np.random.default_rng(seed)

  precs, lifts = [], []
  for _ in range(n_boot):
    elegidos = rng.choice(len(claves), size=len(claves), replace=True)
    idx = np.concatenate([grupos[claves[i]] for i in elegidos])
    yi, fi = y[idx], f[idx]

    tasa_base_i = yi.mean()
    if fi.sum() > 0 and tasa_base_i > 0:
      prec_i = ((fi == 1) & (yi == 1)).sum() / fi.sum()
      precs.append(prec_i)
      lifts.append(prec_i / tasa_base_i)

  fmt_lift = (
      f"[{np.percentile(lifts, 2.5):.2f} - {np.percentile(lifts, 97.5):.2f}]"
      if len(lifts) >= 50
      else "N/A"
  )
  fmt_prec = (
      f"[{np.percentile(precs, 2.5):.2f} - {np.percentile(precs, 97.5):.2f}]"
      if len(precs) >= 50
      else "N/A"
  )
  return fmt_prec, fmt_lift


tabla_critica = []

for ciudad, cfg in CONFIG.items():
  d = df[(df["city"] == ciudad) & df["month"].isin([5, 6, 7, 8])].copy()
  if cfg["desde"]:
    d = d[d["date_local"] >= cfg["desde"]]

  d["y"] = (d["pm25_next_day"] >= cfg["target"]).astype(int)
  d["alerta_hoy"] = (d["pm25_mean_day"] >= cfg["target"]).astype(int)
  d = d.dropna(subset=FEATURES_LIN + ["y", "alerta_hoy"]).reset_index(drop=True)

  tr = d[d["date_local"] < cfg["fecha_test"]].copy()
  te = d[d["date_local"] >= cfg["fecha_test"]].copy()

  clf = make_pipeline(
      StandardScaler(),
      LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000),
  ).fit(tr[FEATURES_LIN], tr["y"])

  probs_tr = clf.predict_proba(tr[FEATURES_LIN])[:, 1]
  umbral_30 = float(np.quantile(probs_tr, 0.70))

  te["prob"] = clf.predict_proba(te[FEATURES_LIN])[:, 1]
  te["pred"] = (te["prob"] >= umbral_30).astype(int)

  y_true = te["y"].to_numpy()
  pred = te["pred"].to_numpy()

  tp = int(((pred == 1) & (y_true == 1)).sum())
  fp = int(((pred == 1) & (y_true == 0)).sum())
  fn = int(((pred == 0) & (y_true == 1)).sum())
  n_test = len(te)
  n_alertas_reales = int(y_true.sum())

  # Métricas base y Lift
  tasa_base = n_alertas_reales / n_test
  precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
  lift = precision / tasa_base if tasa_base > 0 else 0.0
  ratio_fp_tp = fp / tp if tp > 0 else np.nan

  ic_prec, ic_lift = bootstrap_lift(te)

  tabla_critica.append({
      "Ciudad": ciudad,
      "Días Test": n_test,
      "Tasa Base (Azar)": f"{tasa_base:.1%}",
      "Precisión Modelo": f"{precision:.1%} {ic_prec}",
      "Lift vs Azar": f"{lift:.2f}x {ic_lift}",
      "Aciertos (TP)": tp,
      "Falsas Alarmas (FP)": fp,
      "FP por Acierto": f"{ratio_fp_tp:.1f} FP / TP",
      "Veredicto Operativo": (
          "Excelente (duplica azar)"
          if lift >= 1.8
          else "Marginal (cerca del azar)"
      ),
  })

df_res_critico = pd.DataFrame(tabla_critica)

print("\n" + "=" * 125)
print("  ANÁLISIS CRÍTICO DE RENDIMIENTO: PRECISION VS TASA BASE (LIFT SOBRE EL AZAR)")
print("=" * 125)
print(df_res_critico.to_string(index=False))
print("=" * 125)

# Guardar en processed para respaldo
out_critico_csv = PROCESSED_DIR / "analisis_critico_lift_test.csv"
df_res_critico.to_csv(out_critico_csv, index=False)
print(f"Resumen guardado en: {out_critico_csv}")


  ANÁLISIS CRÍTICO DE RENDIMIENTO: PRECISION VS TASA BASE (LIFT SOBRE EL AZAR)
  Ciudad  Días Test Tasa Base (Azar)    Precisión Modelo        Lift vs Azar  Aciertos (TP)  Falsas Alarmas (FP) FP por Acierto       Veredicto Operativo
Santiago        241            17.4% 22.2% [0.11 - 0.34] 1.28x [0.90 - 1.72]             16                   56    3.5 FP / TP Marginal (cerca del azar)
  Temuco        230            36.5% 77.0% [0.64 - 0.88] 2.11x [1.70 - 2.68]             47                   14    0.3 FP / TP  Excelente (duplica azar)
   Aysén        246            39.0% 80.7% [0.68 - 0.90] 2.07x [1.68 - 2.82]             71                   17    0.2 FP / TP  Excelente (duplica azar)
Valdivia        123            31.7% 59.4% [0.39 - 0.76] 1.87x [1.38 - 2.50]             19                   13    0.7 FP / TP  Excelente (duplica azar)
Resumen guardado en: C:\Users\chest\OneDrive\Documentos\Portafolio\Medicion contaminantes\data\processed\analisis_critico_lift_test.csv


In [13]:
# ==============================================================================
# TABLA MAESTRA DEFINITIVA CORREGIDA: BENCHMARK FUERA DE MUESTRA E IC 95%
# ==============================================================================
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CURRENT_DIR = Path().resolve()
BASE_DIR = (
    CURRENT_DIR
    if (CURRENT_DIR / "data" / "processed").exists()
    else CURRENT_DIR.parent
)
PROCESSED_DIR = BASE_DIR / "data" / "processed"
gold_path = PROCESSED_DIR / "daily_modeling_gold_prueba_final.parquet"

df = pd.read_parquet(gold_path)
df["date_local"] = pd.to_datetime(df["date_local"])

FEATURES_LIN = [
    "temp_min_day_forecast",
    "wind_mean_day_forecast",
    "temperature_2m_noche_forecast",
    "wind_speed_10m_noche_forecast",
    "delta_temp_noche",
    "delta_wind_noche",
    "estancamiento_manana",
]

CONFIG = {
    "Santiago": {"desde": None, "fecha_test": "2025-01-01", "target": 50.0},
    "Temuco": {"desde": None, "fecha_test": "2025-01-01", "target": 50.0},
    "Aysén": {"desde": None, "fecha_test": "2025-01-01", "target": 80.0},
    "Valdivia": {
        "desde": "2025-04-01",
        "fecha_test": "2026-01-01",
        "target": 50.0,
    },
}


def bootstrap_metricas_completas(te_df, n_boot=500, seed=42):
  """Bootstrap por bloques semanales para intervalos IC 95% conjuntos."""
  te = te_df.reset_index(drop=True)
  y = te["y"].to_numpy()
  f = te["pred"].to_numpy()
  p = te["prob"].to_numpy()
  # CORRECCIÓN CLAVE: forzar tipo bool para indexación booleana real
  ini = te["es_primer_dia"].to_numpy().astype(bool)

  grupos = te.groupby(te["date_local"].dt.to_period("W").astype(str)).indices
  claves = list(grupos)
  rng = np.random.default_rng(seed)

  aucs, precs, lifts, recs_ini = [], [], [], []

  for _ in range(n_boot):
    elegidos = rng.choice(len(claves), size=len(claves), replace=True)
    idx = np.concatenate([grupos[claves[i]] for i in elegidos])
    yi, fi, pi, inii = y[idx], f[idx], p[idx], ini[idx]

    tasa_base_i = yi.mean()

    if len(np.unique(yi)) > 1:
      aucs.append(roc_auc_score(yi, pi))

    if fi.sum() > 0 and tasa_base_i > 0:
      prec_i = ((fi == 1) & (yi == 1)).sum() / fi.sum()
      precs.append(prec_i)
      lifts.append(prec_i / tasa_base_i)

    if inii.sum() > 0:
      recs_ini.append(fi[inii].mean())

  fmt = lambda v: (
      f"[{np.percentile(v, 2.5):.2f} - {np.percentile(v, 97.5):.2f}]"
      if len(v) >= 50
      else "N/A"
  )
  low_lift = np.percentile(lifts, 2.5) if len(lifts) >= 50 else np.nan

  return fmt(aucs), fmt(precs), fmt(lifts), fmt(recs_ini), low_lift


tabla_maestra = []

for ciudad, cfg in CONFIG.items():
  d = df[(df["city"] == ciudad) & df["month"].isin([5, 6, 7, 8])].copy()
  if cfg["desde"]:
    d = d[d["date_local"] >= cfg["desde"]]

  target_val = cfg["target"]
  d["y"] = (d["pm25_next_day"] >= target_val).astype(int)

  # Primer día estricto: dato válido previo, hoy < umbral y mañana >= umbral
  d["es_primer_dia"] = (
      d["pm25_mean_day"].notna()
      & (d["pm25_mean_day"] < target_val)
      & (d["pm25_next_day"] >= target_val)
  ).astype(int)

  d = d.dropna(subset=FEATURES_LIN + ["y"]).reset_index(drop=True)

  tr = d[d["date_local"] < cfg["fecha_test"]].copy()
  te = d[d["date_local"] >= cfg["fecha_test"]].copy()

  clf = make_pipeline(
      StandardScaler(),
      LogisticRegression(C=0.5, class_weight="balanced", max_iter=1000),
  ).fit(tr[FEATURES_LIN], tr["y"])

  probs_tr = clf.predict_proba(tr[FEATURES_LIN])[:, 1]
  umbral_30 = float(np.quantile(probs_tr, 0.70))

  te["prob"] = clf.predict_proba(te[FEATURES_LIN])[:, 1]
  te["pred"] = (te["prob"] >= umbral_30).astype(int)

  y_true = te["y"].to_numpy()
  pred = te["pred"].to_numpy()
  n_test = len(te)

  tp = int(((pred == 1) & (y_true == 1)).sum())
  fp = int(((pred == 1) & (y_true == 0)).sum())
  n_alertas_reales = int(y_true.sum())

  n_primer_dia = int(te["es_primer_dia"].sum())
  tp_primer_dia = int(pred[te["es_primer_dia"] == 1].sum())
  rec_ini_val = (
      tp_primer_dia / n_primer_dia if n_primer_dia > 0 else np.nan
  )

  auc_val = roc_auc_score(y_true, te["prob"])
  tasa_base = n_alertas_reales / n_test
  prec_val = tp / (tp + fp) if (tp + fp) > 0 else 0.0
  lift_val = prec_val / tasa_base if tasa_base > 0 else 0.0
  ratio_fp_tp = fp / tp if tp > 0 else np.nan

  ic_auc, ic_prec, ic_lift, ic_rec_ini, low_lift = (
      bootstrap_metricas_completas(te)
  )

  if low_lift < 1.0:
    veredicto = "No distinguible del azar"
  elif low_lift >= 1.6:
    veredicto = "Cerca del doble del azar"
  else:
    veredicto = "Mejora sobre azar, incierta"

  tabla_maestra.append({
      "Ciudad": ciudad,
      "Días Test": n_test,
      "Tasa Base": f"{tasa_base:.1%}",
      "AUC ROC": f"{auc_val:.3f} {ic_auc}",
      "Precisión": f"{prec_val:.1%} {ic_prec}",
      "Lift vs Azar": f"{lift_val:.2f}x {ic_lift}",
      "1er Día (Recall)": (
          f"{tp_primer_dia}/{n_primer_dia} ({rec_ini_val:.1%}) {ic_rec_ini}"
      ),
      "FP / TP": f"{ratio_fp_tp:.1f}",
      "Veredicto": veredicto,
  })

df_maestra = pd.DataFrame(tabla_maestra)

print("\n" + "=" * 145)
print(
    "  TABLA MAESTRA CONSOLIDADA — EVALUACIÓN FUERA DE MUESTRA E IC 95% BOOTSTRAP"
)
print("=" * 145)
print(df_maestra.to_string(index=False))
print("=" * 145)

out_path = PROCESSED_DIR / "tabla_maestra_evaluacion_final.csv"
df_maestra.to_csv(out_path, index=False)
print(f"Tabla guardada en: {out_path}")


  TABLA MAESTRA CONSOLIDADA — EVALUACIÓN FUERA DE MUESTRA E IC 95% BOOTSTRAP
  Ciudad  Días Test Tasa Base             AUC ROC           Precisión        Lift vs Azar            1er Día (Recall) FP / TP                   Veredicto
Santiago        241     17.4% 0.617 [0.51 - 0.72] 22.2% [0.11 - 0.34] 1.28x [0.90 - 1.72]  5/16 (31.2%) [0.12 - 0.48]     3.5    No distinguible del azar
  Temuco        230     36.5% 0.851 [0.78 - 0.91] 77.0% [0.64 - 0.88] 2.11x [1.70 - 2.68] 15/26 (57.7%) [0.38 - 0.75]     0.3    Cerca del doble del azar
   Aysén        246     39.0% 0.883 [0.84 - 0.93] 80.7% [0.68 - 0.90] 2.07x [1.68 - 2.82] 18/33 (54.5%) [0.37 - 0.74]     0.2    Cerca del doble del azar
Valdivia        123     31.7% 0.746 [0.63 - 0.85] 59.4% [0.39 - 0.76] 1.87x [1.38 - 2.50]  9/17 (52.9%) [0.31 - 0.74]     0.7 Mejora sobre azar, incierta
Tabla guardada en: C:\Users\chest\OneDrive\Documentos\Portafolio\Medicion contaminantes\data\processed\tabla_maestra_evaluacion_final.csv


In [14]:
from pathlib import Path
import joblib

MODELS_DIR = Path("models")
if not MODELS_DIR.exists():
  MODELS_DIR = Path("../models")

for f in sorted(MODELS_DIR.glob("*.joblib")):
  obj = joblib.load(f)
  print(f"Archivo: {f.name}")
  print(f"  • tag_version : {obj.get('tag_version')}")
  print(f"  • gold_source : {obj.get('gold_source')}")
  print(f"  • training    : {obj.get('training_window')}")
  print("-" * 50)

Archivo: alert_model_aysen_v2.joblib
  • tag_version : v2
  • gold_source : daily_modeling_gold_prueba_final.parquet
  • training    : 2021-05-01 a 2026-08-31
--------------------------------------------------
Archivo: alert_model_aysén_v2.joblib
  • tag_version : v2
  • gold_source : None
  • training    : None
--------------------------------------------------
Archivo: alert_model_santiago_v2.joblib
  • tag_version : v2
  • gold_source : daily_modeling_gold_prueba_final.parquet
  • training    : 2021-05-01 a 2026-08-31
--------------------------------------------------
Archivo: alert_model_temuco_v2.joblib
  • tag_version : v2
  • gold_source : daily_modeling_gold_prueba_final.parquet
  • training    : 2021-05-01 a 2026-08-31
--------------------------------------------------
Archivo: alert_model_valdivia_v2.joblib
  • tag_version : v2
  • gold_source : daily_modeling_gold_prueba_final.parquet
  • training    : 2025-05-01 a 2026-08-31
-------------------------------------------------